# Sellers - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim, create_map, lit

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_sellers"
target_table = f"{catalog}.silver.olist_sellers"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

seller_id,seller_zip_code_prefix,seller_city,seller_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 3095
Number of columns: 11


In [0]:
columns = ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


for column in columns:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())
    print("Extra whitespace row count:",
        (
            bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
            .filter(col(column) != col(f"{column}_trimmed"))
            .count()
        )
    )
    print("-"*20)

seller_id
Null count: 0
Distinct count: 3095
Extra whitespace row count: 0
--------------------
seller_zip_code_prefix
Null count: 0
Distinct count: 2246
Extra whitespace row count: 0
--------------------
seller_city
Null count: 0
Distinct count: 611
Extra whitespace row count: 0
--------------------
seller_state
Null count: 0
Distinct count: 23
Extra whitespace row count: 0
--------------------


- seller_id does not contain any null value and its distinct count is equal to the number of rows in the table. Hence it is a valid key.
- There are no null values or extra whitespace in any column.

In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


There are no rescued data.

## Transform to Silver

In [0]:
brazil_state_map = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AM": "Amazonas",
    "AP": "Amapá",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MG": "Minas Gerais",
    "MS": "Mato Grosso do Sul",
    "MT": "Mato Grosso",
    "PA": "Pará",
    "PB": "Paraíba",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "PR": "Paraná",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RO": "Rondônia",
    "RR": "Roraima",
    "RS": "Rio Grande do Sul",
    "SC": "Santa Catarina",
    "SE": "Sergipe",
    "SP": "São Paulo",
    "TO": "Tocantins"
}

In [0]:
state_map_expr = create_map(
    *[
        item
        for state_code, state_name in brazil_state_map.items()
        for item in (lit(state_code), lit(state_name))
    ]
)

silver_df = bronze_df.withColumn(
    "seller_state_name",
    state_map_expr[col("seller_state")]
)

In [0]:
display(
    silver_df.select("seller_state", "seller_state_name").distinct()
)

seller_state,seller_state_name
SP,São Paulo
RJ,Rio de Janeiro
PE,Pernambuco
PR,Paraná
GO,Goiás
SC,Santa Catarina
BA,Bahia
DF,Distrito Federal
RS,Rio Grande do Sul
MG,Minas Gerais


## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)
 |-- seller_state_name: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

seller_id,seller_zip_code_prefix,seller_city,seller_state,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset,seller_state_name
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,Rio de Janeiro
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,Rio de Janeiro
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,Pernambuco
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,São Paulo
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,null,/Volumes/ecommerce_dev/landing/raw_files/olist/sellers/olist_sellers_dataset.csv,2026-08-02T21:30:44.000Z,2026-08-02T23:03:40.306Z,e7a1225a-74e0-477f-944e-c722d6871604,olist,sellers,Paraná


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 3095
Silver row count: 3095
